# Goal #2 — Patent Classification (Abstract-only)

This notebook implements **Goal #2** from the project instructions:

- Merge the provided US patent CSV exports.
- Classify each patent into the predefined ferrofluids taxonomy.
- Output **Primary / Secondary / Tertiary** class codes and descriptions.
- Include a **Reasoning** string for each row.
- **Critical constraint:** classification is based **exclusively on the Abstract** (not Title / CPC / metadata).

Inputs in this repo:
- `raw_data/MANI_KW_PATENTS_A_weds1969to2009.csv`
- `raw_data/MANI_KW_PATENTS_B_weds2010tonow.csv`
- `classification/FEROFLUIDS_CLASS_DEFINITION.md`
- `docs/INSTRUCTIONS.md`
- `docs/AI_CLASSIFICATON_IS421_2026S.md`

Outputs:
- `outputs/goal2_patents_classified.csv` (sorted by Year → Primary → Secondary → Tertiary; preserves all original columns)

In [ ]:
# Mount Drive (Google Colab)
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/mani

## 1) Runtime Setup (paths, installs, reproducibility)

Colab free tier + zero-cost OSS tooling only. This section installs dependencies, sets paths, and defines a shared `CONFIG` dict.

In [ ]:
# Dependency setup for Colab (avoid breaking Colab's pinned scientific stack)
#
# NOTE:
# - Do NOT `pip -U` core packages like numpy/pandas in Colab.
# - If you previously upgraded them and hit import errors, this cell will
#   pin back to compatible versions ONCE and then restart the runtime.

import os
import re
import json
import sys
import subprocess
from pathlib import Path


def pip_install(*args: str) -> None:
    cmd = [sys.executable, '-m', 'pip', 'install', '-q', *args]
    subprocess.check_call(cmd)


SENTINEL = Path('/content/.mani_deps_ok')

if not SENTINEL.exists():
    # Pin to Colab-compatible versions to avoid SciPy / TF / other conflicts.
    # (Colab currently expects pandas==2.2.2 and numpy < 2.2.)
    pip_install('pandas==2.2.2', 'numpy<2.2', 'scipy<1.13')

    # Install Hugging Face runtime deps.
    pip_install('-U', 'transformers', 'accelerate', 'sentencepiece', 'tqdm')

    SENTINEL.write_text('ok', encoding='utf-8')
    print('Installed/pinned dependencies. Restarting runtime...')
    os._exit(0)
else:
    # Normal path on subsequent runs (no restarts).
    pip_install('-U', 'transformers', 'accelerate', 'sentencepiece', 'tqdm')

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
from transformers import pipeline

# Reproducibility
SEED = 421
np.random.seed(SEED)
torch.manual_seed(SEED)

ROOT = Path('.').resolve()

CONFIG = {
    'patents_csv_a': str(ROOT / 'raw_data' / 'MANI_KW_PATENTS_A_weds1969to2009.csv'),
    'patents_csv_b': str(ROOT / 'raw_data' / 'MANI_KW_PATENTS_B_weds2010tonow.csv'),
    'class_md': str(ROOT / 'classification' / 'FEROFLUIDS_CLASS_DEFINITION.md'),
    'out_dir': str(ROOT / 'outputs'),
    'out_csv': str(ROOT / 'outputs' / 'goal2_patents_classified.csv'),
    'model_name': 'typeform/distilbert-base-uncased-mnli',
    'batch_size': 16,
    'hypothesis_template': 'This text is about {}.',
}

Path(CONFIG['out_dir']).mkdir(parents=True, exist_ok=True)
print('ROOT:', ROOT)
print('Versions:', {'numpy': np.__version__, 'pandas': pd.__version__})
print(json.dumps(CONFIG, indent=2))

## 2) Load & Display Assignment Instructions from `docs/`

These are included in-notebook for traceability and to keep constraints visible while you work.

In [ ]:
from IPython.display import Markdown, display

instructions_path = ROOT / 'docs' / 'INSTRUCTIONS.md'
assignment_path = ROOT / 'docs' / 'AI_CLASSIFICATON_IS421_2026S.md'

instructions_text = instructions_path.read_text(encoding='utf-8', errors='ignore')
assignment_text = assignment_path.read_text(encoding='utf-8', errors='ignore')

display(Markdown('## docs/INSTRUCTIONS.md'))
display(Markdown(instructions_text))

display(Markdown('## docs/AI_CLASSIFICATON_IS421_2026S.md'))
display(Markdown(assignment_text))

## 3) Goal #2 Config: Parse/Record Requirements (from docs)

We capture the Goal #2 requirements into a machine-readable JSON file for reproducibility.

In [ ]:
goal2_requirements = {
    'goal': 'Goal #2 — Classification of US Patents',
    'data_sources': [
        'raw_data/MANI_KW_PATENTS_A_weds1969to2009.csv',
        'raw_data/MANI_KW_PATENTS_B_weds2010tonow.csv',
    ],
    'classification_basis': 'Abstract only (Title/Keywords/metadata must not influence classification)',
    'required_outputs': [
        'Serial Number',
        'Year',
        'Title',
        'Primary Class',
        'Primary Desc',
        'Secondary Class',
        'Secondary Desc',
        'Tertiary Class',
        'Tertiary Desc',
        'Reasoning',
    ],
    'integrity_constraint': 'All original columns must remain in final dataset',
    'sorting': ['Year', 'Primary Class', 'Secondary Class', 'Tertiary Class'],
    'taxonomy_source': 'classification/FEROFLUIDS_CLASS_DEFINITION.md',
    'modeling_constraints': {
        'no_paid_apis': True,
        'open_source_huggingface': True,
        'colab_free_tier': True,
    },
}

req_path = ROOT / 'goal2_requirements.json'
req_path.write_text(json.dumps(goal2_requirements, indent=2), encoding='utf-8')
print(f'Wrote requirements JSON: {req_path}')
req_path

## 4) Dataset Ingest: Merge Patent CSVs, Validate Schema, Clean

For Goal #2 we load both patent CSV exports, merge them, validate required columns, clean obvious non-data rows, and generate a manual `Serial Number`.

Train/validation/test splits are **not applicable** here because we are not training a supervised model (we are applying abstract-only zero-shot classification).

In [ ]:
pat_a = Path(CONFIG['patents_csv_a'])
pat_b = Path(CONFIG['patents_csv_b'])
for p in [pat_a, pat_b]:
    if not p.exists():
        raise FileNotFoundError(f'Missing patent CSV: {p}')

df_a = pd.read_csv(pat_a, dtype=str)
df_b = pd.read_csv(pat_b, dtype=str)
df_raw = pd.concat([df_a, df_b], ignore_index=True)
print('A shape:', df_a.shape)
print('B shape:', df_b.shape)
print('Merged shape:', df_raw.shape)

required_input_cols = ['Publication Year', 'Title', 'Abstract']
missing_input = [c for c in required_input_cols if c not in df_raw.columns]
if missing_input:
    raise ValueError(f'Missing required columns in patent CSVs: {missing_input}')

# Normalize to standard names used elsewhere in the project
df = df_raw.copy()
df['Year'] = df['Publication Year']
df['Year_num'] = pd.to_numeric(df['Year'], errors='coerce')
df['Abstract_clean'] = df['Abstract'].fillna('').astype(str).str.strip()

before = len(df)
df = df[df['Year_num'].notna() & (df['Abstract_clean'].str.len() > 0)].copy()
print('Dropped rows:', before - len(df))
print('Clean rows:', len(df))

# Manual serial number (requirement)
df.insert(0, 'Serial Number', np.arange(1, len(df) + 1))
df['Year_num'] = df['Year_num'].astype(int)

df[['Serial Number', 'Year', 'Title']].head()

## 5) Preprocessing Pipeline

Preprocessing for Goal #2 is primarily:
- parsing the class taxonomy from the provided markdown
- ensuring **only** abstract text is fed to the classifier
- basic text cleanup (trim whitespace)

No one-hot encoding / scaling needed for this goal.

In [ ]:
class_md_path = Path(CONFIG['class_md'])
if not class_md_path.exists():
    raise FileNotFoundError(f'Missing class definition markdown: {class_md_path}')

class_text = class_md_path.read_text(encoding='utf-8', errors='ignore')

def parse_class_definitions(markdown_text: str):
    classes = []
    current = None

    for raw in markdown_text.splitlines():
        line = raw.rstrip('\n')
        stripped = line.strip()

        # Start of a class row: two-digit code
        if re.match(r'^\s*\d{2}\s', line):
            if current is not None:
                classes.append(current)

            parts = re.split(r'\s{2,}', stripped, maxsplit=2)
            if len(parts) == 3:
                code, major, desc = parts
            elif len(parts) == 2:
                code, major = parts
                desc = ''
            else:
                continue

            current = {'code': int(code), 'major': major.strip(), 'desc': desc.strip()}
            continue

        # Ignore separators / headers
        if not stripped:
            continue
        if stripped.startswith('---') or stripped.startswith('[') or stripped.startswith('MAJOR CLASSES'):
            continue
        if re.match(r'^\d+\.', stripped):
            continue

        # Continuation line for multi-line descriptions
        if current is not None:
            if set(stripped) <= set('-|'):
                continue
            current['desc'] = (current['desc'] + ' ' + stripped).strip()

    if current is not None:
        classes.append(current)

    out = []
    for c in classes:
        out.append({
            'code': int(c['code']),
            'major': re.sub(r'\s+', ' ', c['major']).strip(),
            'desc': re.sub(r'\s+', ' ', c['desc']).strip(),
        })

    out = sorted(out, key=lambda x: x['code'])

    # Label for model: include code + major + desc to keep mapping unambiguous
    for c in out:
        if c['desc']:
            c['label'] = f"{c['code']} {c['major']} - {c['desc']}"
        else:
            c['label'] = f"{c['code']} {c['major']}"

    return out

classes = parse_class_definitions(class_text)
print('Parsed classes:', len(classes))
pd.DataFrame(classes).head(10)

## 6) Classifier: Abstract-only Zero-shot Classification

Goal #2 does not require supervised training (no labeled ground truth is provided). Instead, we apply an **abstract-only zero-shot classifier** (open-source Hugging Face model) using the predefined class labels as candidate labels.

In [ ]:
MODEL_NAME = CONFIG['model_name']

# Device selection
device = 0 if torch.cuda.is_available() else -1
print('CUDA available:', torch.cuda.is_available(), '| device:', device)

zero_shot = pipeline(
    task='zero-shot-classification',
    model=MODEL_NAME,
    device=device,
)

candidate_labels = [c['label'] for c in classes]
label_to_meta = {c['label']: c for c in classes}

print('Initialized zero-shot model:', MODEL_NAME)
print('Candidate labels:', len(candidate_labels))
print('Example label:', candidate_labels[0])

In [ ]:
def _top3_from_result(result: dict) -> dict:
    labels = result['labels'][:3]
    scores = result['scores'][:3]
    metas = [label_to_meta[l] for l in labels]

    primary = metas[0]
    secondary = metas[1] if len(metas) > 1 else metas[0]
    tertiary = metas[2] if len(metas) > 2 else metas[-1]

    reasoning = (
        'Abstract-only zero-shot classification. '
        + 'Top-3 labels by score: '
        + ', '.join([f"{l} ({s:.3f})" for l, s in zip(labels, scores)])
    )

    return {
        'Primary Class': primary['code'],
        'Primary Desc': primary['desc'] if primary['desc'] else primary['major'],
        'Secondary Class': secondary['code'],
        'Secondary Desc': secondary['desc'] if secondary['desc'] else secondary['major'],
        'Tertiary Class': tertiary['code'],
        'Tertiary Desc': tertiary['desc'] if tertiary['desc'] else tertiary['major'],
        'Reasoning': reasoning,
        'Top1_score': float(scores[0]) if len(scores) > 0 else None,
        'Top2_score': float(scores[1]) if len(scores) > 1 else None,
    }


def classify_abstracts_zero_shot(texts: list[str], batch_size: int):
    # IMPORTANT: We feed ONLY abstract text here.
    outputs = []
    for i in tqdm(range(0, len(texts), batch_size)):
        batch = texts[i : i + batch_size]
        res = zero_shot(
            batch,
            candidate_labels=candidate_labels,
            multi_label=False,
            hypothesis_template=CONFIG['hypothesis_template'],
            truncation=True,
        )
        if isinstance(res, dict):
            res = [res]
        outputs.extend(res)
    return outputs

batch_size = int(CONFIG['batch_size'])
abstracts = df['Abstract_clean'].tolist()

print(f'Starting zero-shot classification for {len(abstracts)} patent abstracts (batch_size={batch_size})')
results = classify_abstracts_zero_shot(abstracts, batch_size=batch_size)

classified_rows = [_top3_from_result(r) for r in results]
classified_df = pd.DataFrame(classified_rows)
print('Finished zero-shot classification')

classified_df.head(3)

## 7) Metrics & Diagnostics (no ground truth)

Goal #2 does not provide ground-truth labels, so we cannot compute accuracy/F1/confusion matrix.

Instead we compute **sanity metrics** useful for review:
- distribution of Primary Class
- low-confidence / ambiguous classifications (small Top1–Top2 margin)

These diagnostics do not affect the exported CSV deliverable.

In [ ]:
# Primary class distribution
primary_counts = classified_df['Primary Class'].value_counts(dropna=False).sort_index()
primary_counts_df = primary_counts.rename('count').reset_index().rename(columns={'index': 'Primary Class'})
primary_counts_df.head(20)

# Ambiguity score: margin between top1 and top2
classified_df['margin_12'] = classified_df['Top1_score'] - classified_df['Top2_score']
ambiguous = classified_df.nsmallest(25, 'margin_12')[['Primary Class', 'Secondary Class', 'Top1_score', 'Top2_score', 'margin_12', 'Reasoning']]
ambiguous

# Save metrics for future review
metrics_dir = Path(CONFIG['out_dir'])
primary_counts_path = metrics_dir / 'goal2_metrics_primary_class_counts.csv'
ambiguous_path = metrics_dir / 'goal2_metrics_ambiguous_top25.csv'
primary_counts_df.to_csv(primary_counts_path, index=False)
ambiguous.to_csv(ambiguous_path, index=False)
print('Saved metrics:', primary_counts_path)
print('Saved metrics:', ambiguous_path)

## 8) Iteration: Spot-check Refinement Pass (BART on Ambiguous Rows)

Optional: rerun the most ambiguous rows using a heavier MNLI model for a spot-check (trade runtime for quality).

In [ ]:
# Spot-check refinement pass: run BART on the most ambiguous rows only.
#
# Prereqs: run Sections 4–7 first so you have df + classified_df + candidate_labels.
# Output: writes a comparison CSV to outputs/ for manual review.
#
# IMPORTANT:
# - We keep BART predictions index-aligned to the original rows when applying refinement.
# - We only sort a COPY for display/export to avoid misalignment bugs.

ALT_MODEL_NAME = 'facebook/bart-large-mnli'
REFINE_N = 200            # how many ambiguous rows to rerun with BART
ALT_BATCH_SIZE = 8        # reduce to 4 if you hit GPU OOM
APPLY_REFINEMENT = True   # set False for review-only; True overwrites those rows in classified_df

if 'margin_12' not in classified_df.columns:
    classified_df['margin_12'] = classified_df['Top1_score'] - classified_df['Top2_score']

# Select lowest-margin rows (most ambiguous)
refine_idx = (
    classified_df
    .dropna(subset=['margin_12'])
    .nsmallest(int(REFINE_N), 'margin_12')
    .index
    .tolist()
 )

print(f'Refining {len(refine_idx)} ambiguous rows with: {ALT_MODEL_NAME}')

device = 0 if torch.cuda.is_available() else -1
zero_shot_bart = pipeline(
    task='zero-shot-classification',
    model=ALT_MODEL_NAME,
    device=device,
 )

def classify_with_pipeline(pipe, texts: list[str], batch_size: int):
    outputs = []
    for i in tqdm(range(0, len(texts), batch_size)):
        batch = texts[i : i + batch_size]
        res = pipe(
            batch,
            candidate_labels=candidate_labels,
            multi_label=False,
            hypothesis_template=CONFIG['hypothesis_template'],
            truncation=True,
        )
        if isinstance(res, dict):
            res = [res]
        outputs.extend(res)
    return outputs

refine_texts = df.loc[refine_idx, 'Abstract_clean'].tolist()
bart_results = classify_with_pipeline(zero_shot_bart, refine_texts, batch_size=int(ALT_BATCH_SIZE))

# Keep BART predictions index-aligned for safe application
bart_pred = pd.DataFrame([_top3_from_result(r) for r in bart_results], index=refine_idx)

# Pretty/explicit columns for spot-check export
bart_df = bart_pred.rename(columns={
    'Primary Class': 'BART Primary Class',
    'Primary Desc': 'BART Primary Desc',
    'Secondary Class': 'BART Secondary Class',
    'Secondary Desc': 'BART Secondary Desc',
    'Tertiary Class': 'BART Tertiary Class',
    'Tertiary Desc': 'BART Tertiary Desc',
    'Reasoning': 'BART Reasoning',
    'Top1_score': 'BART Top1_score',
    'Top2_score': 'BART Top2_score',
})
bart_df['BART margin_12'] = bart_df['BART Top1_score'] - bart_df['BART Top2_score']

# Build a comparison table (baseline vs BART)
base_cols = [
    'Primary Class', 'Secondary Class', 'Tertiary Class',
    'Top1_score', 'Top2_score', 'margin_12',
 ]
base_subset = classified_df.loc[refine_idx, base_cols].copy()
base_subset = base_subset.rename(columns={
    'Primary Class': 'Baseline Primary Class',
    'Secondary Class': 'Baseline Secondary Class',
    'Tertiary Class': 'Baseline Tertiary Class',
    'Top1_score': 'Baseline Top1_score',
    'Top2_score': 'Baseline Top2_score',
    'margin_12': 'Baseline margin_12',
})

meta_subset = df.loc[refine_idx, ['Serial Number', 'Year', 'Title']].copy()

comparison = pd.concat([meta_subset, base_subset, bart_df], axis=1)
comparison['Primary agrees'] = (comparison['Baseline Primary Class'] == comparison['BART Primary Class'])

# Sort ONLY for viewing/export (do not reset index for alignment-sensitive operations)
comparison_sorted = comparison.sort_values(by=['Baseline margin_12'], ascending=True)

out_path = Path(CONFIG['out_dir']) / 'goal2_bart_refinement_spotcheck.csv'
comparison_sorted.to_csv(out_path, index=False)
print('Wrote spot-check refinement table:', out_path)
comparison_sorted.head(20)

if APPLY_REFINEMENT:
    # Overwrite only the refined rows in classified_df (keeps baseline elsewhere).
    apply_cols = [
        'Primary Class', 'Primary Desc',
        'Secondary Class', 'Secondary Desc',
        'Tertiary Class', 'Tertiary Desc',
        'Reasoning', 'Top1_score', 'Top2_score',
    ]
    for col in apply_cols:
        classified_df.loc[refine_idx, col] = bart_pred.loc[refine_idx, col].to_numpy()

    classified_df['margin_12'] = classified_df['Top1_score'] - classified_df['Top2_score']
    print('Applied refinement to classified_df for', len(refine_idx), 'rows.')

## 9) Error/Edge-case Analysis

Without ground truth, we treat **low-margin** cases (Top1≈Top2) as edge cases that are worth manual review.

The table below joins those edge cases back to the original rows for inspection.

In [ ]:
edge_idx = ambiguous.index.tolist()
edge_cases = pd.concat(
    [df.loc[edge_idx, ['Serial Number', 'Year', 'Title', 'Abstract']].reset_index(drop=True),
     ambiguous.reset_index(drop=True)],
    axis=1,
)
edge_cases

## 10) Save Artifacts

Saves:
- the final sorted CSV for Goal #2
- a small JSON snapshot with configuration
- optional review tables (edge cases)

In [ ]:
# Assemble final output while preserving ALL original columns
export_class_cols = [
    'Primary Class', 'Primary Desc',
    'Secondary Class', 'Secondary Desc',
    'Tertiary Class', 'Tertiary Desc',
    'Reasoning',
]
classified_export = classified_df[export_class_cols].copy()

df_out = pd.concat([df.reset_index(drop=True), classified_export.reset_index(drop=True)], axis=1)

# Sort by: Year, Primary, Secondary, Tertiary
for col in ['Primary Class', 'Secondary Class', 'Tertiary Class']:
    df_out[col] = pd.to_numeric(df_out[col], errors='coerce')

df_out = df_out.sort_values(
    by=['Year_num', 'Primary Class', 'Secondary Class', 'Tertiary Class'],
    ascending=True,
).reset_index(drop=True)

# Remove helper-only columns (keep all original input columns)
df_out = df_out.drop(columns=['Year_num', 'Abstract_clean'], errors='ignore')

out_csv_path = Path(CONFIG['out_csv'])
df_out.to_csv(out_csv_path, index=False)

# Save config snapshot
config_path = Path(CONFIG['out_dir']) / 'goal2_run_config.json'
config_path.write_text(json.dumps(CONFIG, indent=2), encoding='utf-8')

# Save edge cases
edge_path = Path(CONFIG['out_dir']) / 'goal2_edge_cases.csv'
edge_cases.to_csv(edge_path, index=False)

print('Wrote:', out_csv_path)
print('Also wrote:', config_path)
print('Also wrote:', edge_path)
print('Rows:', len(df_out))

# Required columns check
required_cols = goal2_requirements['required_outputs']
missing = [c for c in required_cols if c not in df_out.columns]
print('Missing required columns:', missing)

df_out[required_cols].head(5)